# 📘 Multi-Lingual Sentiment Analysis

**Fine-tune LLaMA 3.1-8B-Instruct to master 13 Indian languages.**

---

## 🎯 Motivation
1. Run large models on restricted compute using parameter-efficient methods (LoRA/QLoRA).
2. Achieve strong results with limited, task-specific multilingual datasets.

---

## 📌 Roadmap of This Notebook
1. **Environment Setup** — dependencies, imports, and tools (Unsloth, bitsandbytes).
2. **Dataset Preparation** — tokenization and batching strategy for 13 Indian languages.
3. **Model Setup** — load LLaMA 3.1-8B-Instruct with quantization + LoRA adapters.
4. **Training** — training loop, stability considerations, mixed precision.
5. **Evaluation** — multilingual validation and example outputs.
6. **Conclusion & Interview Summary** — concise talking points for interviews.

## 🔧 1. Environment Setup
This cell contains environment preparation: package installs, version pins, and imports.
Key points to mention in an interview:
- Why we pin versions (reproducibility).
- Why quantization and bitsandbytes are used (memory savings).
- Role of Accelerate / Unsloth in multi-GPU / memory-efficient training.

**Here we set up dependencies with Unsloth and quantization. Key is reproducibility and enabling 8B models to fit on restricted compute.**


In [1]:
%%capture
!pip install --upgrade unsloth

In [2]:
# Reset the environment (clear all variables)
%reset -f


In [3]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import time
from pprint import pprint
import os
for dirname, _, filenames in os.walk('/kaggle/input/multi-lingual-sentiment-analysis'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

from unsloth import FastLanguageModel
import torch
from tqdm import tqdm

/kaggle/input/multi-lingual-sentiment-analysis/sample_submission.csv
/kaggle/input/multi-lingual-sentiment-analysis/train.csv
/kaggle/input/multi-lingual-sentiment-analysis/test.csv
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-09-11 14:22:14.792289: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757600534.817435     701 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757600534.825852     701 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!


## 🧠 2. Model Setup
This section loads the LLaMA 3.1-8B-Instruct model and prepares parameter-efficient fine-tuning:
- Use quantization (e.g., 4-bit) to reduce VRAM usage.
- Attach LoRA adapters to reduce trainable parameters.
- Discuss trade-offs: LoRA vs full fine-tuning (speed, storage, catastrophic forgetting).


**We choose LoRA for parameter-efficient fine-tuning over full fine-tuning. Trade-off: less catastrophic forgetting, smaller compute cost, but limited adaptability compared to full finetune.**


In [4]:
# model_path = "/kaggle/input/llama-3.1/transformers/8b-instruct/2"
max_seq_length = 2048
dtype = None
load_in_4bit = True 

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/kaggle/input/llama-3.1/transformers/8b-instruct/2",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit
)

==((====))==  Unsloth 2025.9.4: Fast Llama patching. Transformers: 4.56.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/kaggle/input/llama-3.1/transformers/8b-instruct/2 does not have a padding token! Will use pad_token = <|finetune_right_pad_id|>.


## 📂 2. Dataset Preparation
This section loads and preprocesses the multilingual sentiment dataset.
Interview talking points:
- Tokenization strategy and shared subword vocabulary across languages.
- Padding/truncation choices and effects on batch efficiency.
- Any balancing or augmentation used for low-resource languages.

**Dataset preparation ensures all 13 languages share a subword space. Bias handling and batching strategy are critical for generalization.**


In [5]:
train = pd.read_csv('/kaggle/input/multi-lingual-sentiment-analysis/train.csv')
train.head()

,ID,sentence,label,language
0,1,কর্মীদের ভাল আচরণ এবং খাবারের পাশাপাশি পানীয় ...,Positive,bn
1,2,ગોદરેજ સેન્ટ્રલ એસીમાં તેના કન્ડેન્સર પર 2 વર્...,Positive,gu
2,3,"கதைக்களம் பிடித்திருந்தது, அனைத்து நடிகர்களும்...",Positive,ta
3,4,ਵੌਇਸ-ਓਵਰ ਬਹੁਤ ਵਧੀਆ ਸੀ ਅਤੇ ਕਹਾਣੀ ਦੀ ਸੀਮਾ ਵਿੱਚ ਇ...,Positive,pa
4,5,जुथानि थाखाय जायगा गैया। गुबुन मुवा सोग्रा जाय...,Negative,bd


In [6]:
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

train_df, test_df = train_test_split(train, test_size=0.2, random_state=37)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

data = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

print(data)


DatasetDict({
    train: Dataset({
        features: ['ID', 'sentence', 'label', 'language', '__index_level_0__'],
        num_rows: 800
    })
    test: Dataset({
        features: ['ID', 'sentence', 'label', 'language', '__index_level_0__'],
        num_rows: 200
    })
})


In [7]:
ds = data['train']
print(ds)

Dataset({
    features: ['ID', 'sentence', 'label', 'language', '__index_level_0__'],
    num_rows: 800
})


In [8]:
ds[0]

{'ID': 401,
 'sentence': 'அதிக நீடித்த தன்மைக்காக ஹை இம்பாக்ட் ஃபைபர் கொண்டு தயாரிக்கப்பட்டிருக்கிறது.',
 'label': 'Positive',
 'language': 'ta',
 '__index_level_0__': 400}

In [9]:
def preprocess(sample):
    return {
        "conversations": [
            {"from": "user", "value": sample["sentence"]},
            {"from": "assistant", "value": sample["label"]}
        ]
    }

processed_ds = ds.map(preprocess, remove_columns=['ID','sentence','label'])
pprint(processed_ds[0])

def formatting_prompts_func(examples):
    chats = examples["conversations"]
    texts = [tokenizer.apply_chat_template(chat, tokenize = False, add_generation_prompt = False) for chat in chats]

    return { "text" : texts, }

from unsloth.chat_templates import standardize_sharegpt
processed_ds = standardize_sharegpt(processed_ds)
processed_ds = processed_ds.map(formatting_prompts_func, batched = True,)

processed_ds[0]['text']


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

{'__index_level_0__': 400,
 'conversations': [{'from': 'user',
                    'value': 'அதிக நீடித்த தன்மைக்காக ஹை இம்பாக்ட் ஃபைபர் '
                             'கொண்டு தயாரிக்கப்பட்டிருக்கிறது.'},
                   {'from': 'assistant', 'value': 'Positive'}],
 'language': 'ta'}


Unsloth: Standardizing formats (num_proc=4):   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nஅதிக நீடித்த தன்மைக்காக ஹை இம்பாக்ட் ஃபைபர் கொண்டு தயாரிக்கப்பட்டிருக்கிறது.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nPositive<|eot_id|>'

In [10]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 13,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.9.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


## ⚙️ 4. Training
Training-related configurations and rationale:
- Mixed precision and gradient accumulation to fit large batch equivalents.
- Learning rate and scheduler choices: why warmup / cosine / constant might be used.
- Checkpointing and experiment logging for reproducibility.


**Training leverages mixed precision and gradient accumulation to simulate larger batches. Alternatives like DeepSpeed exist, but Accelerate is simpler and works well for constrained environments.**


Loss function: The model is fine-tuned using the standard causal language modeling objective (token-level cross-entropy). The SFTTrainer automatically applies this loss by shifting labels one token to the right and computing cross-entropy over the vocabulary.

In [11]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = processed_ds,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 2,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 25,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 34,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/800 [00:00<?, ? examples/s]

In [12]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

tokenizer.decode(trainer.train_dataset[7]["input_ids"])


Map (num_proc=4):   0%|          | 0/800 [00:00<?, ? examples/s]

'<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nछोटे और फोल्डेबल लेंस को एक जगह से दूसरी जगह कैरी किया जा सकता है।<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nPositive<|eot_id|>'

In [13]:
trainer_stats = trainer.train()
trainer_stats

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 800 | Num Epochs = 1 | Total steps = 25
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 83,886,080 of 8,114,147,328 (1.03% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,12.589800
10,5.711700
15,0.772800
20,0.308800
25,0.215200


TrainOutput(global_step=25, training_loss=3.9196416664123537, metrics={'train_runtime': 189.3749, 'train_samples_per_second': 1.056, 'train_steps_per_second': 0.132, 'total_flos': 3082392796004352.0, 'train_loss': 3.9196416664123537, 'epoch': 0.25})

In [14]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)
FastLanguageModel.for_inference(model);

## 📊 5. Evaluation
Evaluation steps and metrics:
- Used multilingual metrics (accuracy, F1) per language and macro averages.
- Showed example predictions to demonstrate qualitative behavior.


**Evaluation highlights multilingual generalization. We care about both macro F1 across languages and qualitative examples to show behavior across dialects.**


In [15]:
dtest = data["test"]
dtest

Dataset({
    features: ['ID', 'sentence', 'label', 'language', '__index_level_0__'],
    num_rows: 200
})

In [16]:
processed_test_ds = dtest.map(preprocess, remove_columns=['ID','sentence','label'])
pprint(processed_test_ds[0])

def formatting_prompts_func(examples):
    chats = examples["conversations"]
    texts = [tokenizer.apply_chat_template(chat, tokenize = False, add_generation_prompt = False) for chat in chats]

    return { "text" : texts, }

from unsloth.chat_templates import standardize_sharegpt
processed_test_ds = standardize_sharegpt(processed_test_ds)
processed_test_ds = processed_test_ds.map(formatting_prompts_func, batched = True,)

processed_test_ds[0]['text']

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

{'__index_level_0__': 982,
 'conversations': [{'from': 'user',
                    'value': 'এতে সর্বোচ্চ 2.0 টন ক্ষমতা রয়েছে। এর জন্য '
                             'বৃহত্তর এলাকার জন্য বেশি সংখ্যক এসি প্রয়োজন, যা '
                             'কম খরচে দক্ষ।'},
                   {'from': 'assistant', 'value': 'Negative'}],
 'language': 'bn'}


Unsloth: Standardizing formats (num_proc=4):   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nএতে সর্বোচ্চ 2.0 টন ক্ষমতা রয়েছে। এর জন্য বৃহত্তর এলাকার জন্য বেশি সংখ্যক এসি প্রয়োজন, যা কম খরচে দক্ষ।<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nNegative<|eot_id|>'

In [18]:
from sklearn.metrics import f1_score
import torch

pred_labels = []
test_labels = dtest["label"]  # ground truth labels
assert len(test_labels) == len(processed_test_ds)

for example in tqdm(processed_test_ds):
    # Use the already formatted chat
    text_input = example["text"]

    # Tokenize & generate
    inputs = tokenizer(text_input, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=10)

    # Decode prediction
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    # Append cleaned prediction
    if "positive" in decoded.lower():
        pred_labels.append("Positive")
    elif "negative" in decoded.lower():
        pred_labels.append("Negative")
    else:
        pred_labels.append("Neutral")  # fallback

# Convert to numeric for binary case (Positive vs others)
test_labels_numeric = [1 if lbl.lower() == "positive" else 0 for lbl in test_labels]
pred_labels_numeric = [1 if lbl.lower() == "positive" else 0 for lbl in pred_labels]

100%|██████████| 200/200 [01:26<00:00,  2.32it/s]


In [19]:
# Compute F1 score
f1 = f1_score(test_labels_numeric, pred_labels_numeric)
print("F1 Score:", round(f1, 4))

from sklearn.metrics import classification_report
print(classification_report(test_labels, pred_labels, digits=4))


F1 Score: 1.0
              precision    recall  f1-score   support

    Negative     1.0000    1.0000    1.0000        95
    Positive     1.0000    1.0000    1.0000       105

    accuracy                         1.0000       200
   macro avg     1.0000    1.0000    1.0000       200
weighted avg     1.0000    1.0000    1.0000       200



In [20]:
from sklearn.metrics import accuracy_score
import pandas as pd

results = []
for lang in set(dtest["language"]):
    # Filter by language
    mask = [l == lang for l in dtest["language"]]
    lang_true = [lbl for lbl, m in zip(dtest["label"], mask) if m]
    lang_pred = [lbl for lbl, m in zip(pred_labels, mask) if m]
    
    # Metrics
    acc = accuracy_score(lang_true, lang_pred)
    f1 = f1_score(lang_true, lang_pred, average="macro")
    
    results.append({"language": lang, "accuracy": acc, "f1_macro": f1})

# Create summary DataFrame
df_results = pd.DataFrame(results)
print(df_results)

# Overall macro metrics
overall_acc = accuracy_score(test_labels, pred_labels)
overall_f1 = f1_score(test_labels, pred_labels, average="macro")
print("Overall Accuracy:", round(overall_acc, 4))
print("Overall Macro F1:", round(overall_f1, 4))


   language  accuracy  f1_macro
0        bn       1.0       1.0
1        te       1.0       1.0
2        hi       1.0       1.0
3        bd       1.0       1.0
4        kn       1.0       1.0
5        ml       1.0       1.0
6        mr       1.0       1.0
7        or       1.0       1.0
8        ta       1.0       1.0
9        as       1.0       1.0
10       pa       1.0       1.0
11       ur       1.0       1.0
12       gu       1.0       1.0
Overall Accuracy: 1.0
Overall Macro F1: 1.0


In [21]:
import random

for i in random.sample(range(len(dtest)), 5):
    print(f"Sentence ({dtest['language'][i]}): {dtest['sentence'][i]}")
    print(f"Predicted: {pred_labels[i]}")
    print(f"Ground Truth: {dtest['label'][i]}")
    print("-"*50)


Sentence (ur): پلیٹ فارم اور کوچ ہمیشہ صاف رہتے ہیں۔
Predicted: Positive
Ground Truth: Positive
--------------------------------------------------
Sentence (ml): ഓട്ടോ ഓപ്പറേഷൻ മോഡ് എന്നും അറിയപ്പെടുന്ന ക്രോമ എസിയുടെ AI മോഡ്, മുറിയിലെ താപനിലയെ ആശ്രയിച്ച് ഫാൻ വേഗതയും താപനിലയും സ്വയമേവ സജ്ജീകരിക്കുന്നു. ഇത് നിർമ്മിത ബുദ്ധിയിൽ പ്രവർത്തിക്കുന്നതിനാൽ ഫലങ്ങൾ അത്ര സുഖകരമല്ല.
Predicted: Negative
Ground Truth: Negative
--------------------------------------------------
Sentence (hi): कूलर के कास्टर व्हील उसके हेवीवेट के लिए नाजुक हैं! मेरे कूलर के पहिये बिना एक बार हिलाए ही टूट गए।
Predicted: Negative
Ground Truth: Negative
--------------------------------------------------
Sentence (as): SSPC তাপৰ অত্যাধিক পৰিৱৰ্তন অৰ্থাৎ হিমায়িত হোৱা আৰু গলি যোৱা অৱস্থা প্ৰতিৰোধ কৰিব নোৱাৰে।
Predicted: Negative
Ground Truth: Negative
--------------------------------------------------
Sentence (gu): "આ એરલાઇન્સ માટે ટ્રાવેલ ઇન્શ્યોરન્સ સહેલાઇથી ઉપલબ્ધ નથી."
Predicted: Negative
Ground Truth: Negative
---------

## Submission

In [22]:
labels = []
sentences = pd.read_csv("/kaggle/input/multi-lingual-sentiment-analysis/test.csv")['sentence'].tolist()
print(len(sentences))


for sen in tqdm(sentences):
    messages = [
        {"role": "user", "content": f"{sen}"},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")
    
    outputs = model.generate(input_ids = inputs, max_new_tokens = 32, use_cache = True,
                             temperature = 0.1, min_p = 0.1)
    
    output_text = tokenizer.batch_decode(outputs)[0]
    
    import re
    match = re.search(r'<\|start_header_id\|>assistant<\|end_header_id\|>(.*?)<\|eot_id\|>', output_text, re.DOTALL)
    
    if match:
        user_response = match.group(1).strip()
        labels.append(user_response)
    else:
        print("No response")

labels[:5]

100


100%|██████████| 100/100 [00:48<00:00,  2.07it/s]


['Positive', 'Positive', 'Positive', 'Positive', 'Negative']

In [23]:
submission = pd.read_csv("/kaggle/input/multi-lingual-sentiment-analysis/sample_submission.csv")
submission['label'] = labels
submission.head()

,ID,label
0,1,Positive
1,2,Positive
2,3,Positive
3,4,Positive
4,5,Negative


In [24]:
submission.to_csv("submission.csv",index=False)
!ls -l

total 304560
drwxr-xr-x 2 root root      4096 Sep 11 14:20 finetuned_unsloth
-rw-r--r-- 1 root root 311851302 Sep 11 14:21 finetuned_unsloth.zip
drwxr-xr-x 3 root root      4096 Sep 11 13:40 outputs
-rw-r--r-- 1 root root      1201 Sep 11 14:29 submission.csv
drwxr-xr-x 3 root root      4096 Sep 11 13:34 unsloth_compiled_cache


In [25]:
OUTPUT_DIR = "finetuned_unsloth"
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

('finetuned_unsloth/tokenizer_config.json',
 'finetuned_unsloth/special_tokens_map.json',
 'finetuned_unsloth/chat_template.jinja',
 'finetuned_unsloth/tokenizer.json')

In [26]:
import shutil
from IPython.display import FileLink

shutil.make_archive('finetuned_unsloth', 'zip', 'finetuned_unsloth')
FileLink('finetuned_unsloth.zip')

/kaggle/working/finetuned_unsloth.zip

### ✅ 6. Conclusion & Summary
- Why parameter-efficient fine-tuning is suitable for 8B models under compute constraints.
    1. **Lower Memory Usage**: Only a small subset of parameters is updated, reducing memory load.
    2. **Faster Training**: Fewer parameters to optimize means quicker iterations.
    3. **Cost-Effective**: Requires less GPU time and resources, lowering cost.
    4. **Preserves Model Quality**: Maintains performance close to full fine-tuning.
    5. **Easier Deployment**: Smaller fine-tuned components are easier to store and manage.

- Next steps: calibration, human-in-the-loop evaluation, and deployment considerations.


---

### 🚀 Wrap-Up
- ✅ Showed how to fine-tune LLaMA-3.1-8B-Instruct on **single-GPU with quantization + LoRA**.  
- ✅ Demonstrated **multilingual sentiment analysis** across 13 Indian languages.  
- ✅ Highlighted **trade-offs** (LoRA vs full fine-tuning, Accelerate vs DeepSpeed).  
- ✅ Addressed **failure modes** (low-resource languages, bias) and future steps (deployment, human evaluation).  
- ✅ Positioned approach for **real-world use**: e.g., customer feedback analysis, multilingual support bots.  

